# Question 5 — Do shape traits track environmental (climate) change?

**Question**: are morphological shape traits (sphericity, shape factor) more or less responsive to climate change than size traits are? We test this by correlating shape traits against benthic δ18O / δ13C isotope records, which serve as a proxy for past ocean temperature and carbon cycling.

**Pipeline**:
1. Load the merged morphometric + environmental dataset
2. Visualise shape traits and climate proxies as time series, and their pairwise relationships
3. Quantify the relationship with a correlation matrix and regression models

## Setup

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

## Load data

`merged_data.xlsx` already combines the morphometric mastersheet with the benthic isotope records, so we load it directly rather than merging separate files.

In [ ]:
df = pd.read_excel(DATA_DIR / "merged_data.xlsx")

In [ ]:
columns_to_keep = [
    'Age_Ma',
    'Size.Mean.Sphericity',
    'Size.Mean.ShapeFactor',
    #'benthic d18O VPDB',    
    #'benthic d13C VPDB',     
    'benthic d18O smoothLoess10',
    'benthic d13C smoothLoess10'
]

In [ ]:
df_clean = df[columns_to_keep].dropna()

In [ ]:
df_clean = df_clean.sort_values('Age_Ma')

In [ ]:
print(df_clean.head())

## Time series and smoothed relationships

First, plot sphericity/shape factor and the climate proxies (δ18O/δ13C) over geological time. Then look at their pairwise relationships directly with LOWESS-smoothed scatterplots.

In [ ]:
# Create 2x2 grid layout
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Time Series of Shape and Environmental Parameters', fontsize=16)

# --- Plot 1: Sphericity over Time ---
axs[0, 0].plot(df_clean['Age_Ma'], df_clean['Size.Mean.Sphericity'], color='blue')
axs[0, 0].invert_xaxis()
axs[0, 0].set_title('Sphericity Over Time')
axs[0, 0].set_xlabel('Age (Ma)')
axs[0, 0].set_ylabel('Sphericity')
axs[0, 0].grid(True)

# --- Plot 2: Shape Factor over Time ---
axs[0, 1].plot(df_clean['Age_Ma'], df_clean['Size.Mean.ShapeFactor'], color='green')
axs[0, 1].invert_xaxis()
axs[0, 1].set_title('Shape Factor Over Time')
axs[0, 1].set_xlabel('Age (Ma)')
axs[0, 1].set_ylabel('Shape Factor')
axs[0, 1].grid(True)

# --- Plot 3: Smoothed δ18O over Time ---
axs[1, 0].plot(df_clean['Age_Ma'], df_clean['benthic d18O smoothLoess10'], color='red')
axs[1, 0].invert_xaxis()
axs[1, 0].set_title('Smoothed δ18O Over Time')
axs[1, 0].set_xlabel('Age (Ma)')
axs[1, 0].set_ylabel('δ18O (‰)')
axs[1, 0].grid(True)

# --- Plot 4: Smoothed δ13C over Time ---
axs[1, 1].plot(df_clean['Age_Ma'], df_clean['benthic d13C smoothLoess10'], color='purple')
axs[1, 1].invert_xaxis()
axs[1, 1].set_title('Smoothed δ13C Over Time')
axs[1, 1].set_xlabel('Age (Ma)')
axs[1, 1].set_ylabel('δ13C (‰)')
axs[1, 1].grid(True)

# Final layout tweak
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# Set up a 2x2 grid of scatter plots with LOWESS smoothing
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Smoothed Relationships Between Shape Traits and Environmental Proxies", fontsize=16)

# Plot 1: Sphericity vs δ18O
sns.regplot(ax=axs[0, 0], x='benthic d18O smoothLoess10', y='Size.Mean.Sphericity',
            data=df_clean, lowess=True,
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'blue'})
axs[0, 0].set_title('Sphericity vs δ18O')
axs[0, 0].grid(True)

# Plot 2: Shape Factor vs δ18O
sns.regplot(ax=axs[0, 1], x='benthic d18O smoothLoess10', y='Size.Mean.ShapeFactor',
            data=df_clean, lowess=True,
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'green'})
axs[0, 1].set_title('Shape Factor vs δ18O')
axs[0, 1].grid(True)

# Plot 3: Sphericity vs δ13C
sns.regplot(ax=axs[1, 0], x='benthic d13C smoothLoess10', y='Size.Mean.Sphericity',
            data=df_clean, lowess=True,
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'orange'})
axs[1, 0].set_title('Sphericity vs δ13C')
axs[1, 0].grid(True)

# Plot 4: Shape Factor vs δ13C
sns.regplot(ax=axs[1, 1], x='benthic d13C smoothLoess10', y='Size.Mean.ShapeFactor',
            data=df_clean, lowess=True,
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'purple'})
axs[1, 1].set_title('Shape Factor vs δ13C')
axs[1, 1].grid(True)

# Layout adjustment
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Quantifying the relationship

Correlation matrix and regression models between shape traits and climate proxies.

**Finding**: shape-based traits (sphericity, shape factor) showed a clearer relationship to climate proxies than size-based traits did — colder conditions were associated with less spherical, more elongate shell morphology.

In [ ]:
# Select relevant columns
corr_data = df[[
    'Size.Mean.Sphericity',
    'Size.Mean.ShapeFactor',
    'benthic d18O smoothLoess10',
    'benthic d13C smoothLoess10'
]].dropna()

# Rename for clarity
corr_data.columns = [
    'Sphericity',
    'Shape Factor',
    'δ18O (Smooth)',
    'δ13C (Smooth)'
]

# Compute and print a rounded correlation matrix
corr_matrix = corr_data.corr().round(2)

# Print the matrix
print("Correlation Matrix:\n")
print(corr_matrix.to_string())


# Step 3: Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", square=True,
            cbar_kws={"shrink": 0.8}, linewidths=0.5)

plt.title('Correlation Heatmap: Shape Traits vs Environment')
plt.show()

In [ ]:
df = df_clean[[
    'Size.Mean.Sphericity',
    'Size.Mean.ShapeFactor',
    'benthic d18O smoothLoess10',
    'benthic d13C smoothLoess10'
]].dropna()

df.columns = ['Sphericity', 'ShapeFactor', 'd18O', 'd13C']

def print_simple_summary(model, model_name):
    print(f"\n {model_name}")
    print("-" * 40)
    print("Coefficients:")
    print(model.params.round(4))
    print("\nP-values:")
    print(model.pvalues.round(4))
    print(f"\nR²: {model.rsquared:.4f}")
    print("-" * 40)

# --- Model 1: Sphericity ~ d18O + d13C
X1 = sm.add_constant(df[['d18O', 'd13C']])
y1 = df['Sphericity']
model1 = sm.OLS(y1, X1).fit()
print_simple_summary(model1, "Sphericity ~ d18O + d13C")

# --- Model 2: ShapeFactor ~ d18O + d13C
X2 = sm.add_constant(df[['d18O', 'd13C']])
y2 = df['ShapeFactor']
model2 = sm.OLS(y2, X2).fit()
print_simple_summary(model2, "ShapeFactor ~ d18O + d13C")